# Embedding Experiments — Stratified Permutation Tests

Tables are built by `build_correctness_tables.py` and cached in `results/correctness_tables/`.

| # | Analysis | Factor | Stratification |
|---|----------|--------|----------------|
| 1 | **Shot count** | 0 vs 1 vs Few-shot | Model |
| 2 | **Model similarity** | Model comparison | Domain |
| 3 | **Classifier head** | LR vs SVM | Model |
| 4 | **Model classifier** | Model comparison | Domain |

Global **Holm correction** applied across all tests.

## 0. Imports & Setup

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from statsmodels.stats.multitest import multipletests

project_root = os.path.abspath(".")
if project_root not in sys.path: sys.path.insert(0, project_root)
os.chdir(project_root)

from orgpackage.aux import load_experiments
from orgpackage.config import DOMAIN_CLASSES_CORR
from orgpackage.tester import stratified_permutation_test

warnings.filterwarnings("ignore")
CORRECTNESS_DIR = "./results/correctness_tables"
DOMAINS = ["medical", "administrative", "education"]
print("Setup complete.")

## 1. Load Experiments & Correctness Tables

> Tables should have been pre-built by `python build_correctness_tables.py`.

In [ ]:
exps = load_experiments()
emb_exps = exps[exps["Technique"] == "embedding"]
sim_exps  = emb_exps[emb_exps["Method"] == "similarity"]
clf_exps  = emb_exps[emb_exps["Method"] == "classifier"]

# Build indices
sim_index = {}; clf_index = {}
for _, r in sim_exps.iterrows():
    p = r['Parameters']
    if isinstance(p, dict): sim_index.setdefault(p.get('model'), {}).setdefault(p.get('n_shot'), []).append(r['ID'])
for _, r in clf_exps.iterrows():
    p = r['Parameters']
    if isinstance(p, dict):
        t = "lr" if "solver" in p else ("svm" if "kernel" in p else "unknown")
        clf_index.setdefault(p.get('model'), {}).setdefault(t, []).append(r['ID'])

# Load cached correctness tables
sim_ct = {}; clf_ct = {}
for f in os.listdir(CORRECTNESS_DIR):
    if not f.endswith('.csv'): continue
    name = f.replace('.csv', '')
    if name.endswith('_sim'):
        m = name[:-4]; sim_ct[m] = pd.read_csv(os.path.join(CORRECTNESS_DIR, f), index_col=0)
    elif name.endswith('_clf'):
        m = name[:-4]; clf_ct[m] = pd.read_csv(os.path.join(CORRECTNESS_DIR, f), index_col=0)

print(f"Similarity models: {list(sim_ct.keys())}")
print(f"Classifier models: {list(clf_ct.keys())}")

## 2. Run Tests & Plot Results

In [ ]:
def _get_id(index, m, k, d):
    for eid in index.get(m, {}).get(k, []):
        if eid.startswith(d[:3]): return eid
    return None

def run_p_test(strata, lbl, tag):
    if not strata: return pd.DataFrame()
    # Correct call: random_state=42, statistic='pooled'
    obs, pval = stratified_permutation_test(strata, "A", "B",
                                            n_perm=10000,
                                            random_state=42,
                                            statistic="pooled")
    return pd.DataFrame([{"analysis": tag, "comparison": lbl,
                          "obs_diff": obs, "p_value": pval}])

all_res = []

# A1: Shot Count
for sa, sb, lbl in [("1_shot","0_shot","1-shot vs 0-shot"),
                    ("few_shot","1_shot","few vs 1-shot"),
                    ("few_shot","0_shot","few vs 0-shot")]:
    st = []
    for m, ct in sim_ct.items():
        for d in DOMAINS:
            ia, ib = _get_id(sim_index, m, sa, d), _get_id(sim_index, m, sb, d)
            if ia and ib and ia in ct.columns and ib in ct.columns:
                st.append(ct[[ia, ib]].rename(columns={ia: "A", ib: "B"}))
    all_res.append(run_p_test(st, lbl, "A1_shot"))

# A2: Similarity Model Comparison
for sa in ["0_shot", "1_shot", "few_shot"]:
    for ma, mb in combinations(sorted(sim_ct.keys()), 2):
        st = []
        for d in DOMAINS:
            ia, ib = _get_id(sim_index, ma, sa, d), _get_id(sim_index, mb, sa, d)
            if ia and ib and ia in sim_ct[ma].columns and ib in sim_ct[mb].columns:
                jo = sim_ct[ma][[ia]].join(sim_ct[mb][[ib]], how="inner").rename(columns={ia:"A",ib:"B"})
                if not jo.empty: st.append(jo)
        all_res.append(run_p_test(st, f"{ma} vs {mb}", f"A2_model_{sa}"))

# A3: LR vs SVM
st = []
for m, ct in clf_ct.items():
    lr = [_get_id(clf_index, m, "lr", d) for d in DOMAINS]
    sv = [_get_id(clf_index, m, "svm", d) for d in DOMAINS]
    ia = [i for i in lr if i and i in ct.columns]
    ib = [i for i in sv if i and i in ct.columns]
    if ia and ib:
        st.append(pd.concat([ct[ia].astype(float).mean(axis=1),
                             ct[ib].astype(float).mean(axis=1)], axis=1).rename(columns={0:"A",1:"B"}))
all_res.append(run_p_test(st, "LR vs SVM", "A3_clf_head"))

# A4: Classifier Model Comparison
for head in ["lr", "svm"]:
    for ma, mb in combinations(sorted(clf_ct.keys()), 2):
        st = []
        for d in DOMAINS:
            ia, ib = _get_id(clf_index, ma, head, d), _get_id(clf_index, mb, head, d)
            if ia and ib and ia in clf_ct[ma].columns and ib in clf_ct[mb].columns:
                jo = clf_ct[ma][[ia]].join(clf_ct[mb][[ib]], how="inner").rename(columns={ia:"A",ib:"B"})
                if not jo.empty: st.append(jo)
        all_res.append(run_p_test(st, f"{ma} vs {mb}", f"A4_model_{head}"))

print(f"Ran {len(all_res)} tests (including empty). Applying Holm correction…")

## 3. Summary

In [ ]:
final = pd.concat([r for r in all_res if not r.empty], ignore_index=True)
if not final.empty:
    rej, pc, _, _ = multipletests(final['p_value'], method="holm")
    final["p_corrected"] = pc
    final["significant"] = rej
    final.to_csv("results/embedding_permutation_tests.csv", index=False)
    print(f"Saved {len(final)} tests to results/embedding_permutation_tests.csv")

    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    for i, t in enumerate(["A1_shot", "A2_model", "A3_clf_head", "A4_model"]):
        sub = final[final['analysis'].str.startswith(t)]
        if sub.empty: axes[i].set_title(t + "\n(no data)"); continue
        clrs = ["#2ecc71" if (s and d > 0) else ("#e74c3c" if s else "#95a5a6")
                for s, d in zip(sub['significant'], sub['obs_diff'])]
        sns.barplot(data=sub, x="comparison", y="obs_diff", ax=axes[i], palette=clrs)
        axes[i].set_title(t, fontweight='bold')
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].axhline(0, color='black', linewidth=0.8)
        for p_val, bar in zip(sub['p_corrected'], axes[i].patches):
            if p_val < 0.05:
                axes[i].text(bar.get_x() + bar.get_width()/2,
                             bar.get_height(), '*', ha='center', fontsize=14)
    os.makedirs("figures", exist_ok=True)
    plt.tight_layout()
    fig.savefig("figures/perm_tests.png", dpi=300)
    plt.show()
    display(final)
else:
    print("No test results — are the correctness tables loaded?")